<a href="https://colab.research.google.com/github/Haadikp/Image_Colourization/blob/main/Colab_Notebooks/image_colorization_using_gans_unet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U-Net with GAN for Image Colorization

[🔗 Open this notebook in Kaggle](https://www.kaggle.com/code/hadikp/image-colorization-using-gans-unet)

This notebook implements a U-Net-based Generative Adversarial Network (GAN) for image colorization, converting grayscale images (L channel in LAB color space) to colored images by predicting the ab channels. The workflow includes:
- Setting up the environment with Google Colab and GPU support.
- Loading and preprocessing the COCO dataset for training and validation.
- Defining a U-Net generator and a PatchGAN discriminator.
- Training the GAN with a combination of adversarial and L1 loss.
- Saving the trained model.
- Testing the model with online images and evaluating performance using PSNR and SSIM metrics.

The notebook uses PyTorch for model implementation, FastAI for dataset downloading, and Matplotlib for visualization. It leverages the COCO dataset (a subset of 10,000 images) and includes data augmentation for training. The code is executed in Google Colab, as indicated by the metadata.

**Reference**: This implementation is inspired by the [Image Colorization Tutorial by Moein Shariatnia](https://github.com/moein-shariatnia/Deep-Learning/tree/main/Image%20Colorization%20Tutorial).


The below cell mounts your Google Drive to access files directly from the Colab environment. Once mounted, files in Drive can be used for training or loading data/models.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install torchviz

## Import Libraries

This cell imports the necessary Python libraries and modules for the image colorization task.

- **Libraries**:
  - `os`, `glob`, `time`: For file handling and timing.
  - `numpy`: For numerical operations and array manipulation.
  - `PIL.Image`: For image loading and processing.
  - `pathlib.Path`: For filesystem path handling.
  - `tqdm.notebook`: For progress bars in Jupyter/Colab.
  - `matplotlib.pyplot`: For plotting and visualization.
  - `skimage.color.rgb2lab`, `skimage.color.lab2rgb`: For converting images between RGB and LAB color spaces.
  - `torchviz.make_dot`: For visualizing the model's computational graph.
  - `torchsummary.summary`: For printing model summaries.
  - `torch`, `torch.nn`, `torch.optim`: For building and training the PyTorch model.
  - `torchvision.transforms`, `torchvision.utils.make_grid`: For image transformations and visualization utilities.
  - `torch.utils.data.Dataset`, `torch.utils.data.DataLoader`: For creating custom datasets and data loaders.

- **Device Setup**: Defines the `device` variable to use GPU (`cuda`) if available, otherwise CPU.
- **Context**: These imports provide the foundational tools for data processing, model building, training, and evaluation.

In [ ]:
import os
import glob
import time
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from skimage.color import rgb2lab, lab2rgb
from torchviz import make_dot
from torchsummary import summary
import torch
from torch import nn, optim
from torchvision import transforms
from torchvision.utils import make_grid
from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_colab = None

## Install FastAI and Download COCO Dataset

This cell installs the FastAI library (version 2.4) and downloads a subset of the COCO dataset for training and validation.

- **FastAI Installation**: Uses `!pip install fastai==2.4` to install a specific version of FastAI, which provides utilities for dataset downloading.
- **Dataset Download**: Uses `fastai.data.external.untar_data` to download and extract the COCO sample dataset (`URLs.COCO_SAMPLE`), storing it in `coco_path`.
- **Path Setup**:
  - Sets `coco_path` to the `train_sample` subdirectory of the extracted dataset.
  - Defines `use_colab = True` to use the Colab path (`coco_path`) or a custom path if running locally (`use_colab = None`).
- **Output**: The dataset is downloaded to `/root/.fastai/data/coco_sample/train_sample`, containing sample images for training.
- **Context**: The COCO dataset is used for training the colorization model, with images converted to LAB color space later in the notebook.

In [ ]:
!pip install fastai==2.4

In [ ]:
from fastai.data.external import untar_data, URLs
coco_path = untar_data(URLs.COCO_SAMPLE)
coco_path = str(coco_path) + "/train_sample"
use_colab = True

## Prepare Dataset Paths

This cell prepares the dataset by selecting a subset of images from the COCO dataset and splitting them into training and validation sets.

- **Image Selection**:
  - Uses `glob.glob` to collect all `.jpg` files from the dataset directory (`path`).
  - Sets a random seed (`np.random.seed(123)`) for reproducibility.
  - Randomly selects 10,000 images using `np.random.choice`.
- **Train-Validation Split**:
  - Permutes the indices of the 10,000 images using `np.random.permutation`.
  - Assigns the first 8,000 indices to the training set (`train_idxs`) and the last 2,000 to the validation set (`val_idxs`).
  - Creates `train_paths` and `val_paths` lists containing the file paths for training and validation images.
- **Output**: Prints the number of training (8,000) and validation (2,000) images.
- **Context**: This step ensures a manageable dataset size and a clear train-validation split for model training and evaluation.

In [ ]:
if use_colab == True:
    path = coco_path
else:
    path = "Your path to the dataset"

paths = glob.glob(path + "/*.jpg") # Grabbing all the image file names
np.random.seed(123)
paths_subset = np.random.choice(paths, 10_000, replace=False) # choosing 1000 images randomly
rand_idxs = np.random.permutation(10_000)
train_idxs = rand_idxs[:8000] # choosing the first 8000 as training set
val_idxs = rand_idxs[8000:] # choosing last 2000 as validation set
train_paths = paths_subset[train_idxs]
val_paths = paths_subset[val_idxs]
print(len(train_paths), len(val_paths))

## Visualize Sample Images

This cell visualizes a 4x4 grid of sample images from the training set to provide a qualitative sense of the dataset.

- **Visualization**:
  - Creates a 4x4 subplot grid using `plt.subplots` with a figure size of 10x10 inches.
  - Iterates over the first 16 images in `train_paths`, loading each with `PIL.Image.open` and displaying it using `imshow`.
  - Turns off axis labels for cleaner visualization using `ax.axis("off")`.
- **Output**: Displays a grid of 16 RGB images from the training set.
- **Context**: This step helps verify the dataset’s content and quality before preprocessing and training, ensuring the images are suitable for colorization.

In [ ]:
_, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, img_path in zip(axes.flatten(), train_paths):
    ax.imshow(Image.open(img_path))
    ax.axis("off")

## Define Colorization Dataset

This cell defines a custom PyTorch `Dataset` class (`ColorizationDataset`) for loading and preprocessing images for the colorization task.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes a list of image paths (`paths`) and a split type (`train` or `val`).
    - For the training split, applies data augmentation (resize to 256x256 with bicubic interpolation and random horizontal flip).
    - For the validation split, only resizes to 256x256.
    - Stores the split type, image size (`SIZE=256`), and paths.
  - **Data Loading (`__getitem__`)**:
    - Loads an image from the given path using `PIL.Image.open` and converts it to RGB.
    - Applies the defined transformations (resize and optionally flip).
    - Converts the image to LAB color space using `rgb2lab`.
    - Normalizes the L channel to [-1, 1] (dividing by 50 and subtracting 1) and ab channels to [-1, 1] (dividing by 110).
    - Returns a dictionary with `L` (grayscale channel) and `ab` (color channels) as tensors.
  - **Length (`__len__`)**: Returns the number of images in the dataset.
- **Context**: This class prepares images in the LAB color space, where the model takes the L channel as input and predicts the ab channels.

In [ ]:
SIZE = 256
class ColorizationDataset(Dataset):
    def __init__(self, paths, split='train'):
        if split == 'train':
            self.transforms = transforms.Compose([
                transforms.Resize((SIZE, SIZE),  Image.BICUBIC),
                transforms.RandomHorizontalFlip(), # A little data augmentation!
            ])
        elif split == 'val':
            self.transforms = transforms.Resize((SIZE, SIZE),  Image.BICUBIC)

        self.split = split
        self.size = SIZE
        self.paths = paths

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transforms(img)
        img = np.array(img)
        img_lab = rgb2lab(img).astype("float32") # Converting RGB to L*a*b
        img_lab = transforms.ToTensor()(img_lab)
        L = img_lab[[0], ...] / 50. - 1. # Between -1 and 1
        ab = img_lab[[1, 2], ...] / 110. # Between -1 and 1

        return {'L': L, 'ab': ab}

    def __len__(self):
        return len(self.paths)



## Create DataLoaders

This cell defines a utility function `make_dataloaders` to create PyTorch DataLoaders for the training and validation datasets.

- **Function Definition**:
  - Takes parameters for batch size (`batch_size=16`), number of workers (`n_workers=4`), and `pin_memory=True` (for faster GPU data transfer).
  - Accepts additional keyword arguments (`**kwargs`) to pass to the `ColorizationDataset` (e.g., `paths`, `split`).
  - Creates a `ColorizationDataset` instance and wraps it in a `DataLoader` with the specified settings.
- **DataLoader Creation**:
  - Creates `train_dl` for the training set using `train_paths` and `split='train'`.
  - Creates `val_dl` for the validation set using `val_paths` and `split='val'`.
- **Context**: The DataLoaders provide batched data for training and evaluation, with the training DataLoader including data augmentation and the validation DataLoader using only resizing.

In [ ]:
def make_dataloaders(batch_size=16, n_workers=4, pin_memory=True, **kwargs): # A handy function to make our dataloaders
    dataset = ColorizationDataset(**kwargs)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=n_workers,
                            pin_memory=pin_memory)
    return dataloader

## Verify DataLoader Output

This cell verifies the shapes of the data batches produced by the training DataLoader and prints the number of batches in the training and validation DataLoaders.

- **Batch Extraction**: Uses `next(iter(train_dl))` to get the first batch from the training DataLoader, extracting the `L` (grayscale) and `ab` (color) channels.
- **Shape Checking**: Prints the shapes of `Ls` (L channel) and `abs_` (ab channels), expected to be `[batch_size, 1, 256, 256]` and `[batch_size, 2, 256, 256]`, respectively.
- **Batch Count**: Prints the number of batches in `train_dl` (8,000 images / 16 = 500 batches) and `val_dl` (2,000 images / 16 = 125 batches).
- **Context**: This step ensures the DataLoaders are correctly configured and producing data in the expected format before training.

In [ ]:
train_dl = make_dataloaders(paths=train_paths, split='train')
val_dl = make_dataloaders(paths=val_paths, split='val')

data = next(iter(train_dl))
Ls, abs_ = data['L'], data['ab']
print(Ls.shape, abs_.shape)
print(len(train_dl), len(val_dl))

## Define U-Net Block

This cell defines the `UnetBlock` class, a modular component of the U-Net architecture used in the generator.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes parameters for input channels (`input_c`), output filters (`nf`, `ni`), a submodule (for nested U-Net blocks), and flags for `innermost`, `outermost`, and `dropout`.
    - Defines a downsampling path with a convolutional layer (`downconv`, 4x4, stride 2), LeakyReLU, and batch normalization (`downnorm`).
    - Defines an upsampling path with ReLU (`uprelu`), transposed convolution (`upconv`, 4x4, stride 2), and batch normalization (`upnorm`).
    - For the `outermost` block, uses `Tanh` activation and no batch normalization in the upsampling path.
    - For the `innermost` block, omits downsampling normalization.
    - Optionally adds dropout (`nn.Dropout(0.5)`) for intermediate blocks if `dropout=True`.
    - Combines layers into a `nn.Sequential` model.
  - **Forward Pass**: For non-outermost blocks, concatenates the input with the output of the submodule along the channel dimension (skip connections). For the outermost block, returns the model output directly.
- **Context**: This block is the building block of the U-Net, enabling the encoder-decoder structure with skip connections for feature preservation.

In [ ]:
class UnetBlock(nn.Module):
    def __init__(self, nf, ni, submodule=None, input_c=None, dropout=False,
                 innermost=False, outermost=False):
        super().__init__()
        self.outermost = outermost
        if input_c is None: input_c = nf
        downconv = nn.Conv2d(input_c, ni, kernel_size=4,
                             stride=2, padding=1, bias=False)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = nn.BatchNorm2d(ni)
        uprelu = nn.ReLU(True)
        upnorm = nn.BatchNorm2d(nf)

        if outermost:
            upconv = nn.ConvTranspose2d(ni * 2, nf, kernel_size=4,
                                        stride=2, padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(ni, nf, kernel_size=4,
                                        stride=2, padding=1, bias=False)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(ni * 2, nf, kernel_size=4,
                                        stride=2, padding=1, bias=False)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]
            if dropout: up += [nn.Dropout(0.5)]
            model = down + [submodule] + up
        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)



## Define U-Net Generator

This cell defines the `Unet` class, which constructs the full U-Net generator by stacking `UnetBlock` instances.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes parameters for input channels (`input_c=1`, L channel), output channels (`output_c=2`, ab channels), number of downsampling layers (`n_down=8`), and initial filters (`num_filters=64`).
    - Creates the innermost block with 512 filters (`num_filters * 8`).
    - Adds three intermediate blocks with 512 filters and dropout.
    - Adds three blocks with decreasing filters (512, 256, 128).
    - Creates the outermost block to output 2 channels (ab).
    - Combines all blocks into a single model.
  - **Forward Pass**: Passes the input through the outermost block, which recursively processes through all nested blocks.
- **Context**: The U-Net takes a grayscale L channel as input and outputs predicted ab channels, forming the generator for the GAN.

In [ ]:
class Unet(nn.Module):
    def __init__(self, input_c=1, output_c=2, n_down=8, num_filters=64):
        super().__init__()
        unet_block = UnetBlock(num_filters * 8, num_filters * 8, innermost=True)
        for _ in range(n_down - 5):
            unet_block = UnetBlock(num_filters * 8, num_filters * 8, submodule=unet_block, dropout=True)
        out_filters = num_filters * 8
        for _ in range(3):
            unet_block = UnetBlock(out_filters // 2, out_filters, submodule=unet_block)
            out_filters //= 2
        self.model = UnetBlock(output_c, out_filters, input_c=input_c, submodule=unet_block, outermost=True)

    def forward(self, x):
        return self.model(x)

## Define PatchGAN Discriminator

This cell defines the `PatchDiscriminator` class, which implements a PatchGAN discriminator for the GAN framework.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes parameters for input channels (`input_c=3`, L and ab channels), initial filters (`num_filters=64`), and number of downsampling layers (`n_down=3`).
    - Creates a model with:
      - An initial convolutional layer without normalization.
      - `n_down` layers with increasing filters (64, 128, 256, 512), using stride 1 for the last layer.
      - A final layer outputting a single channel (real/fake prediction) without normalization or activation.
    - Uses `get_layers` to create each block with convolution, optional batch normalization, and LeakyReLU.
  - **Forward Pass**: Processes the input through the sequential model, outputting a patch-wise real/fake prediction.
- **Context**: The discriminator evaluates whether an image (L + ab channels) is real or generated, used in the GAN’s adversarial loss.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, input_c, num_filters=64, n_down=3):
        super().__init__()
        model = [self.get_layers(input_c, num_filters, norm=False)]
        model += [self.get_layers(num_filters * 2 ** i, num_filters * 2 ** (i + 1), s=1 if i == (n_down-1) else 2)
                          for i in range(n_down)] # the 'if' statement is taking care of not using
                                                  # stride of 2 for the last block in this loop
        model += [self.get_layers(num_filters * 2 ** n_down, 1, s=1, norm=False, act=False)] # Make sure to not use normalization or
                                                                                             # activation for the last layer of the model
        self.model = nn.Sequential(*model)

    def get_layers(self, ni, nf, k=4, s=2, p=1, norm=True, act=True): # when needing to make some repeatitive blocks of layers,
        layers = [nn.Conv2d(ni, nf, k, s, p, bias=not norm)]          # it's always helpful to make a separate method for that purpose
        if norm: layers += [nn.BatchNorm2d(nf)]
        if act: layers += [nn.LeakyReLU(0.2, True)]
        return nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Test PatchDiscriminator

This cell creates an instance of the `PatchDiscriminator` and tests it with a dummy input to verify its output shape.

- **Instantiation**: Creates a `PatchDiscriminator` with 3 input channels (L + ab).
- **Dummy Input**: Generates a random tensor of shape `[16, 3, 256, 256]` (batch size 16, 3 channels, 256x256 resolution).
- **Forward Pass**: Passes the dummy input through the discriminator and prints the output shape.
- **Output**: The output shape is typically `[16, 1, 16, 16]` (batch size, 1 channel for real/fake, reduced spatial dimensions due to downsampling).
- **Context**: This step verifies that the discriminator is correctly implemented and produces the expected patch-wise output for the GAN.

In [ ]:
PatchDiscriminator(3)
discriminator = PatchDiscriminator(3)
dummy_input = torch.randn(16, 3, 256, 256) # batch_size, channels, size, size
out = discriminator(dummy_input)
out.shape

## Define GAN Loss

This cell defines the `GANLoss` class to compute the adversarial loss for the GAN.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes parameters for GAN mode (`gan_mode='vanilla'` or `'lsgan'`), real label (`real_label=1.0`), and fake label (`fake_label=0.0`).
    - Registers buffers for `real_label` and `fake_label` as tensors.
    - Uses `BCEWithLogitsLoss` for vanilla GAN or `MSELoss` for LSGAN.
  - **Get Labels (`get_labels`)**: Returns labels expanded to match the prediction shape, using `real_label` for real images or `fake_label` for fake images.
  - **Call (`__call__`)**: Computes the loss between predictions and target labels (real or fake).
- **Context**: This class provides the adversarial loss for training the generator and discriminator, with the vanilla GAN mode using binary cross-entropy with logits.

In [ ]:
class GANLoss(nn.Module):
    def __init__(self, gan_mode='vanilla', real_label=1.0, fake_label=0.0):
        super().__init__()
        self.register_buffer('real_label', torch.tensor(real_label))
        self.register_buffer('fake_label', torch.tensor(fake_label))
        if gan_mode == 'vanilla':
            self.loss = nn.BCEWithLogitsLoss()
        elif gan_mode == 'lsgan':
            self.loss = nn.MSELoss()

    def get_labels(self, preds, target_is_real):
        if target_is_real:
            labels = self.real_label
        else:
            labels = self.fake_label
        return labels.expand_as(preds)

    def __call__(self, preds, target_is_real):
        labels = self.get_labels(preds, target_is_real)
        loss = self.loss(preds, labels)
        return loss

## Initialize Model Weights

This cell defines a function to initialize the weights of a PyTorch model.

- **Function Definition (`init_weights`)**:
  - Takes a model (`net`), initialization method (`init='norm'`, `'xavier'`, or `'kaiming'`), and gain (`gain=0.02`).
  - For convolutional layers:
    - Applies normal initialization (`nn.init.normal_`) for `'norm'`.
    - Applies Xavier initialization (`nn.init.xavier_normal_`) for `'xavier'`.
    - Applies Kaiming initialization (`nn.init.kaiming_normal_`) for `'kaiming'`.
    - Initializes biases to 0 if present.
  - For batch normalization layers:
    - Initializes weights with mean 1 and standard deviation `gain`.
    - Initializes biases to 0.
  - Applies the initialization function to all layers using `net.apply`.
  - Prints the initialization method used.
- **Context**: Proper weight initialization is crucial for GAN training stability, with normal initialization used by default.

In [ ]:
def init_weights(net, init='norm', gain=0.02):

    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and 'Conv' in classname:
            if init == 'norm':
                nn.init.normal_(m.weight.data, mean=0.0, std=gain)
            elif init == 'xavier':
                nn.init.xavier_normal_(m.weight.data, gain=gain)
            elif init == 'kaiming':
                nn.init.kaiming_normal_(m.weight.data, a=0, mode='fan_in')

            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0.0)
        elif 'BatchNorm2d' in classname:
            nn.init.normal_(m.weight.data, 1., gain)
            nn.init.constant_(m.bias.data, 0.)

    net.apply(init_func)
    print(f"model initialized with {init} initialization")
    return net

def init_model(model, device):
    model = model.to(device)
    model = init_weights(model)
    return model

## Define Main GAN Model

This cell defines the `MainModel` class, which combines the U-Net generator and PatchGAN discriminator into a GAN framework.

- **Class Definition**:
  - **Initialization (`__init__`)**:
    - Takes parameters for the generator (`net_G`), learning rates (`lr_G`, `lr_D`), Adam optimizer parameters (`beta1`, `beta2`), and L1 loss weight (`lambda_L1=100`).
    - Initializes the generator (U-Net) and discriminator (PatchGAN) using `init_model`.
    - Sets up the GAN loss (`GANLoss`, vanilla mode) and L1 loss (`nn.L1Loss`).
    - Defines Adam optimizers for the generator (`opt_G`) and discriminator (`opt_D`).
  - **Set Requires Grad (`set_requires_grad`)**: Enables or disables gradient computation for a model’s parameters.
  - **Setup Input (`setup_input`)**: Moves input L and ab channels to the device.
  - **Forward Pass**: Generates fake ab channels using the generator.
  - **Backward Discriminator (`backward_D`)**: Computes discriminator loss for real and fake images, averaging them.
  - **Backward Generator (`backward_G`)**: Computes generator loss (adversarial + L1 loss weighted by `lambda_L1`).
  - **Optimize (`optimize`)**: Runs forward pass, updates discriminator, then generator.
- **Context**: This class encapsulates the full GAN training loop, combining adversarial and L1 losses to train the U-Net to produce realistic colorizations.

In [ ]:
class MainModel(nn.Module):
    def __init__(self, net_G=None, lr_G=2e-4, lr_D=2e-4,
                 beta1=0.5, beta2=0.999, lambda_L1=100.):
        super().__init__()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.lambda_L1 = lambda_L1

        if net_G is None:
            self.net_G = init_model(Unet(input_c=1, output_c=2, n_down=8, num_filters=64), self.device)
        else:
            self.net_G = net_G.to(self.device)
        self.net_D = init_model(PatchDiscriminator(input_c=3, n_down=3, num_filters=64), self.device)
        self.GANcriterion = GANLoss(gan_mode='vanilla').to(self.device)
        self.L1criterion = nn.L1Loss()
        self.opt_G = optim.Adam(self.net_G.parameters(), lr=lr_G, betas=(beta1, beta2))
        self.opt_D = optim.Adam(self.net_D.parameters(), lr=lr_D, betas=(beta1, beta2))

    def set_requires_grad(self, model, requires_grad=True):
        for p in model.parameters():
            p.requires_grad = requires_grad

    def setup_input(self, data):
        self.L = data['L'].to(self.device)
        self.ab = data['ab'].to(self.device)

    def forward(self):
        self.fake_color = self.net_G(self.L)

    def backward_D(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        fake_preds = self.net_D(fake_image.detach())
        self.loss_D_fake = self.GANcriterion(fake_preds, False)
        real_image = torch.cat([self.L, self.ab], dim=1)
        real_preds = self.net_D(real_image)
        self.loss_D_real = self.GANcriterion(real_preds, True)
        self.loss_D = (self.loss_D_fake + self.loss_D_real) * 0.5
        self.loss_D.backward()

    def backward_G(self):
        fake_image = torch.cat([self.L, self.fake_color], dim=1)
        fake_preds = self.net_D(fake_image)
        self.loss_G_GAN = self.GANcriterion(fake_preds, True)
        self.loss_G_L1 = self.L1criterion(self.fake_color, self.ab) * self.lambda_L1
        self.loss_G = self.loss_G_GAN + self.loss_G_L1
        self.loss_G.backward()

    def optimize(self):
        self.forward()
        self.net_D.train()
        self.set_requires_grad(self.net_D, True)
        self.opt_D.zero_grad()
        self.backward_D()
        self.opt_D.step()

        self.net_G.train()
        self.set_requires_grad(self.net_D, False)
        self.opt_G.zero_grad()
        self.backward_G()
        self.opt_G.step()

## Define PSNR and SSIM Metrics

This cell imports and defines functions to calculate Peak Signal-to-Noise Ratio (PSNR) and Structural Similarity Index (SSIM) for evaluating the model’s colorization quality.

- **Imports**: Imports `peak_signal_noise_ratio` and `structural_similarity` from `skimage.metrics`.
- **PSNR Function (`calculate_psnr`)**:
  - Takes two RGB images (ground truth and predicted, range [0, 1]).
  - Computes PSNR using `peak_signal_noise_ratio` with `data_range=1.0`.
  - Returns the PSNR value, measuring pixel-wise error.
- **SSIM Function (`calculate_ssim`)**:
  - Takes two RGB images (ground truth and predicted, range [0, 1]).
  - Computes SSIM with `win_size=7`, `channel_axis=-1` (for RGB), and `data_range=1.0`.
  - Returns the SSIM value, measuring structural similarity.
- **Context**: These metrics quantify the quality of the colorized images compared to the ground truth, used in the evaluation phase.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def calculate_psnr(img1, img2):
    """
    Calculate PSNR between two images.
    Args:
        img1: Ground truth image (RGB, range [0, 1]).
        img2: Predicted image (RGB, range [0, 1]).
    Returns:
        PSNR value.
    """
    psnr_value = peak_signal_noise_ratio(img1, img2, data_range=1.0)
    return psnr_value

def calculate_ssim(img1, img2):
    """
    Calculate SSIM between two images.
    Args:
        img1: Ground truth image (RGB, range [0, 1]).
        img2: Predicted image (RGB, range [0, 1]).
    Returns:
        SSIM value.
    """
    # Explicitly set win_size and handle multichannel
    ssim_value = structural_similarity(
        img1, img2, win_size=7, channel_axis=-1, data_range=1.0
    )
    return ssim_value


## Define Average Meter

This cell defines the `AverageMeter` class to track and compute the average of metrics (e.g., losses) during training.

- **Class Definition**:
  - **Initialization (`__init__`)**: Initializes counters for count, sum, and average.
  - **Reset (`reset`)**: Resets all counters to 0.
  - **Update (`update`)**: Updates the sum and count with a new value and batch size, recomputing the average.
- **Context**: Used to monitor training losses (e.g., discriminator fake/real, generator GAN/L1) over epochs, providing a running average for logging.

## Create Loss Meters

This cell defines a function to create a dictionary of `AverageMeter` objects for tracking various losses during training.

- **Function Definition (`create_loss_meters`)**:
  - Creates `AverageMeter` instances for:
    - `loss_D_fake`: Discriminator loss for fake images.
    - `loss_D_real`: Discriminator loss for real images.
    - `loss_D`: Combined discriminator loss.
    - `loss_G_GAN`: Generator adversarial loss.
    - `loss_G_L1`: Generator L1 loss.
    - `loss_G`: Combined generator loss.
  - Returns a dictionary mapping loss names to their respective `AverageMeter` objects.
- **Context**: These meters are used in the training loop to log and monitor the model’s performance across different loss components.

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.count, self.avg, self.sum = [0.] * 3

    def update(self, val, count=1):
        self.count += count
        self.sum += count * val
        self.avg = self.sum / self.count

def create_loss_meters():
    loss_D_fake = AverageMeter()
    loss_D_real = AverageMeter()
    loss_D = AverageMeter()
    loss_G_GAN = AverageMeter()
    loss_G_L1 = AverageMeter()
    loss_G = AverageMeter()

    return {'loss_D_fake': loss_D_fake,
            'loss_D_real': loss_D_real,
            'loss_D': loss_D,
            'loss_G_GAN': loss_G_GAN,
            'loss_G_L1': loss_G_L1,
            'loss_G': loss_G}



## Update Losses

This cell defines a function to update the loss meters with the current batch’s loss values.

- **Function Definition (`update_losses`)**:
  - Takes the model, a dictionary of loss meters (`loss_meter_dict`), and the batch size (`count`).
  - Iterates over the loss meters, retrieving the corresponding loss from the model (e.g., `model.loss_D_fake`) and updating the meter with the loss value and batch size.
- **Context**: Used during training to track the running average of each loss component, enabling periodic logging of training progress.

In [ ]:
def update_losses(model, loss_meter_dict, count):
    for loss_name, loss_meter in loss_meter_dict.items():
        loss = getattr(model, loss_name)
        loss_meter.update(loss.item(), count=count)





## Convert LAB to RGB

This cell defines the `lab_to_rgb` function to convert a batch of LAB images to RGB for visualization and evaluation.

- **Function Definition**:
  - Takes L and ab channel tensors (normalized to [-1, 1]).
  - Denormalizes L (multiply by 50, add 50) and ab (multiply by 110) to their original LAB ranges.
  - Concatenates L and ab channels, permutes to `[batch, height, width, channels]`, and converts to NumPy.
  - Iterates over the batch, converting each LAB image to RGB using `skimage.color.lab2rgb`.
  - Returns a NumPy array of RGB images.
- **Context**: Essential for visualizing the model’s colorized outputs and computing metrics like PSNR and SSIM, which require RGB images.

In [ ]:
def lab_to_rgb(L, ab):
    """
    Takes a batch of images
    """

    L = (L + 1.) * 50.
    ab = ab * 110.
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    rgb_imgs = []
    for img in Lab:
        img_rgb = lab2rgb(img)
        rgb_imgs.append(img_rgb)
    return np.stack(rgb_imgs, axis=0)

## Visualize Model Outputs

This cell defines the `visualize` function to display the model’s colorization results for a batch of images.

- **Function Definition**:
  - Sets the generator to evaluation mode (`model.net_G.eval()`).
  - Processes a batch of data using `model.setup_input` and `model.forward`.
  - Converts predicted (`fake_color`) and ground truth (`ab`) ab channels to RGB using `lab_to_rgb`.
  - Creates a 3x5 plot (3 rows: L channel, fake RGB, real RGB; 5 columns for 5 images).
  - Displays the grayscale L channel, colorized output, and ground truth for each image.
  - Optionally saves the plot as a PNG with a timestamp.
- **Parameters**:
  - `model`: The GAN model.
  - `data`: A batch of data with L and ab channels.
  - `save`: Boolean to save the plot (default `True`).
- **Context**: Used during training to periodically visualize the model’s progress, helping assess the quality of colorization.

In [ ]:
def visualize(model, data, save=True):
    model.net_G.eval()
    with torch.no_grad():
        model.setup_input(data)
        model.forward()
    model.net_G.train()
    fake_color = model.fake_color.detach()
    real_color = model.ab
    L = model.L
    fake_imgs = lab_to_rgb(L, fake_color)
    real_imgs = lab_to_rgb(L, real_color)
    fig = plt.figure(figsize=(15, 8))
    for i in range(5):
        ax = plt.subplot(3, 5, i + 1)
        ax.imshow(L[i][0].cpu(), cmap='gray')
        ax.axis("off")
        ax = plt.subplot(3, 5, i + 1 + 5)
        ax.imshow(fake_imgs[i])
        ax.axis("off")
        ax = plt.subplot(3, 5, i + 1 + 10)
        ax.imshow(real_imgs[i])
        ax.axis("off")
    plt.show()
    if save:
        fig.savefig(f"colorization_{time.time()}.png")

def log_results(loss_meter_dict):
    for loss_name, loss_meter in loss_meter_dict.items():
        print(f"{loss_name}: {loss_meter.avg:.5f}")

## Initialize and Summarize Model

This cell initializes the `MainModel` (GAN) and prints summaries of the generator and discriminator architectures.

- **Model Initialization**: Creates a `MainModel` instance, which includes the U-Net generator and PatchGAN discriminator, and moves it to the device.
- **Model Summaries**:
  - Uses `torchsummary.summary` to print the architecture of the generator (`net_G`) with input shape `(1, 256, 256)` (L channel).
  - Prints the discriminator (`net_D`) summary with input shape `(3, 256, 256)` (L + ab channels).
- **Context**: The summaries provide a detailed view of the model’s layers, parameters, and memory requirements, aiding in debugging and verification.

In [ ]:


# Move model to the appropriate device
model = MainModel().to(device)

# Print the model summary
print("Generator Summary:")
summary(model.net_G, input_size=(1, 256, 256))  # Input: Grayscale image (1 channel)

print("\nDiscriminator Summary:")
summary(model.net_D, input_size=(3, 256, 256))  # Input: L and ab combined (3 channels)


## Visualize Generator Architecture

This cell generates and saves a visual representation of the U-Net generator’s computational graph.

- **Sample Input**: Creates a random tensor of shape `[1, 1, 256, 256]` (single grayscale image) on the device.
- **Forward Pass**: Passes the input through the generator (`model.net_G`) to compute the output.
- **Graph Visualization**: Uses `torchviz.make_dot` to create a computational graph from the output and generator parameters.
- **Output**: Saves the graph as a PNG file named `generator_architecture.png`.
- **Context**: This visualization helps understand the flow of data through the U-Net, useful for debugging and explaining the model’s structure.

In [ ]:


# Generate a sample input for visualization
sample_input = torch.randn(1, 1, 256, 256).to(device)  # Single grayscale image

# Pass the input through the Generator
output = model.net_G(sample_input)

# Visualize the computational graph
graph = make_dot(output, params=dict(model.net_G.named_parameters()))
graph.render("generator_architecture", format="png")  # Save as PNG



## Train the Model

This cell defines and executes the training loop for the GAN model.

- **Function Definition (`train_model`)**:
  - Takes the model, training DataLoader (`train_dl`), number of epochs, and display interval (`display_every=200`).
  - Gets a validation batch for periodic visualization.
  - For each epoch:
    - Creates loss meters to track losses.
    - Iterates over the training DataLoader using `tqdm` for progress tracking.
    - Calls `model.optimize` to update the generator and discriminator.
    - Updates loss meters with `update_losses`.
    - Every `display_every` iterations, prints the epoch, iteration, and losses, and visualizes the model’s output on the validation batch.
- **Execution**:
  - Creates a new `MainModel` instance.
  - Trains the model for 25 epochs.
- **Output**: Prints loss values and displays visualizations every 200 iterations, showing the model’s progress.
- **Context**: This cell trains the GAN to improve the generator’s ability to colorize grayscale images, balancing adversarial and L1 losses.

In [ ]:
def train_model(model, train_dl, epochs, display_every=250):
    data = next(iter(val_dl)) # getting a batch for visualizing the model output after fixed intrvals
    for e in range(epochs):
        loss_meter_dict = create_loss_meters() # function returing a dictionary of objects to
        i = 0                                  # log the losses of the complete network
        for data in tqdm(train_dl):
            model.setup_input(data)
            model.optimize()
            update_losses(model, loss_meter_dict, count=data['L'].size(0)) # function updating the log objects
            i += 1
            if i % display_every == 0:
                print(f"\nEpoch {e+1}/{epochs}")
                print(f"Iteration {i}/{len(train_dl)}")
                log_results(loss_meter_dict) # function to print out the losses
                visualize(model, data, save=False) # function displaying the model's outputs

model = MainModel()
train_model(model, train_dl,15)

## Save the Model

This cell saves the trained model’s weights to Google Drive.

- **Path Setup**: Defines the save directory (`save_path`) on Google Drive and creates it if it doesn’t exist using `os.makedirs`.
- **Model Saving**: Saves the model’s state dictionary to `1:Unet_colorization_model.pth` using `torch.save`.
- **Output**: Prints the file path where the model is saved.
- **Context**: Saving the model allows for later use in testing or deployment without retraining.

In [ ]:
# # Define the path to save the model
# save_path = "/content/drive/MyDrive/saved_model/"

# # Ensure the directory exists
# os.makedirs(save_path, exist_ok=True)

# # Save the model
# model_file_path = os.path.join(save_path, "1:Unet_colorization_model.pth")
# torch.save(model.state_dict(), model_file_path)

# print(f"Model saved at {model_file_path}")

# Define the path to save the model in Kaggle
save_path = "/kaggle/working/"

# Ensure the directory exists
os.makedirs(save_path, exist_ok=True)

# Save the model
model_file_path = os.path.join(save_path, "Unet_colorization_model.pth")
torch.save(model.state_dict(), model_file_path)

print(f"Model saved at {model_file_path}")

## Define Image Download Function

This cell defines a function to download images from URLs for testing the model.

- **Function Definition (`download_image`)**:
  - Takes a URL and uses `requests.get` to fetch the image.
  - If successful (status code 200), opens the image with `PIL.Image.open` and converts it to RGB.
  - Returns the PIL image or `None` if the download fails.
- **Context**: This function enables testing the model with external images, allowing real-world evaluation of the colorization performance.

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def download_image(url):
    """
    Download an image from a URL and return it as a PIL Image.
    """
    response = requests.get(url)
    if response.status_code == 200:
        img = Image.open(BytesIO(response.content)).convert("RGB")
        return img
    else:
        print(f"Failed to download image from {url}")
        return None

def preprocess_image(img, size=256):
    """
    Preprocess the image for the model:
    - Resize to the required size.
    - Convert to L*a*b* and extract the L-channel.
    """
    img = img.resize((size, size), Image.BICUBIC)
    img_array = np.array(img)
    img_lab = rgb2lab(img_array).astype("float32")  # Convert to L*a*b
    L = img_lab[:, :, 0]  # Extract L channel
    L = (L / 50.0) - 1.0  # Normalize L to [-1, 1]
    L_tensor = torch.tensor(L).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions
    return L_tensor.to(device)

def postprocess_and_display(L, ab, original_img):
    """
    Post-process the model's output:
    - Combine L and predicted ab channels.
    - Convert to RGB.
    - Display original grayscale and colorized images side by side.
    """
    L = L.cpu()
    ab = ab.cpu()
    fake_rgb = lab_to_rgb(L, ab)[0]  # Convert first image in batch to RGB
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    # Display Original Image
    ax[0].imshow(original_img)
    ax[0].set_title("Original Image")
    ax[0].axis("off")

    # Display Grayscale (Input)
    ax[1].imshow(L[0, 0], cmap="gray")
    ax[1].set_title("Grayscale Input (L-channel)")
    ax[1].axis("off")

    # Display Colorized (Output)
    ax[2].imshow(fake_rgb)
    ax[2].set_title("Colorized Output")
    ax[2].axis("off")

    plt.show()

def test_model_with_images(model, image_urls, size=256):
    """
    Test the trained model with a list of image URLs.
    Args:
        model: Trained PyTorch model.
        image_urls: List of image URLs.
        size: Size to which images will be resized.
    """
    model.net_G.eval()  # Set the model to evaluation mode
    for url in image_urls:
        print(f"Processing image from: {url}")
        img = download_image(url)  # Download the image
        if img is None:
            continue
        L_tensor = preprocess_image(img, size)  # Preprocess the image
        with torch.no_grad():
            fake_ab = model.net_G(L_tensor)  # Predict ab channels
        postprocess_and_display(L_tensor, fake_ab, img)  # Display results

# List of image URLs
image_urls = [
    "https://images.pexels.com/photos/346529/pexels-photo-346529.jpeg?cs=srgb&dl=pexels-bri-schneiter-28802-346529.jpg&fm=jpg",
    "https://media.istockphoto.com/id/517188688/photo/mountain-landscape.jpg?s=612x612&w=0&k=20&c=A63koPKaCyIwQWOTFBRWXj_PwCrR4cEoOw2S9Q7yVl8=",

]

# Test the model
test_model_with_images(model, image_urls)


## Define Test Function with Metrics

This cell defines the `test_model_with_metrics` function to evaluate the model on the validation set and compute PSNR and SSIM metrics.

- **Function Definition**:
  - Takes the model, validation DataLoader (`val_dl`), and number of samples to visualize (`num_samples=5`).
  - Sets the generator to evaluation mode.
  - Iterates through the validation DataLoader until `num_samples` are visualized:
    - Extracts L and ab channels, moving them to the device.
    - Predicts ab channels (`ab_fake`) using the generator.
    - Converts predicted and ground truth LAB images to RGB using `lab_to_rgb`.
    - Computes PSNR and SSIM for each image pair using `calculate_psnr` and `calculate_ssim`.
    - For the first `num_samples`, displays a 1x3 plot (grayscale input, colorized output, ground truth).
  - Computes and prints the average PSNR and SSIM across all processed samples.
- **Context**: This function provides both quantitative (PSNR, SSIM) and qualitative (visualizations) evaluation of the model’s performance on the validation set.

In [ ]:
def test_model_with_metrics(model, val_dl, num_samples=5):
    """
    Test the model on the test set, visualize results, and calculate PSNR/SSIM.
    Args:
        model: Trained PyTorch model.
        val_dl: DataLoader for the validation/test set.
        num_samples: Number of samples to evaluate and visualize.
    """
    model.net_G.eval()

    psnr_values = []
    ssim_values = []
    samples_shown = 0

    for data in val_dl:
        if samples_shown >= num_samples:
            break

        # Extract inputs and ground truth
        L = data['L'].to(device)
        ab_real = data['ab'].to(device)

        with torch.no_grad():
            # Predict ab channels
            ab_fake = model.net_G(L)

        # Convert to RGB
        L_cpu = L.cpu()
        ab_real_cpu = ab_real.cpu()
        ab_fake_cpu = ab_fake.cpu()

        real_images = lab_to_rgb(L_cpu, ab_real_cpu)  # Ground truth
        fake_images = lab_to_rgb(L_cpu, ab_fake_cpu)  # Predicted

        # Calculate PSNR and SSIM
        for i in range(len(real_images)):
            psnr = calculate_psnr(real_images[i], fake_images[i])
            ssim = calculate_ssim(real_images[i], fake_images[i])
            psnr_values.append(psnr)
            ssim_values.append(ssim)

            if samples_shown < num_samples:
                # Visualize results
                fig, ax = plt.subplots(1, 3, figsize=(15, 5))

                ax[0].imshow(L[i][0].cpu(), cmap="gray")
                ax[0].set_title("Grayscale Input (L-channel)")
                ax[0].axis("off")

                ax[1].imshow(fake_images[i])
                ax[1].set_title("Colorized Output")
                ax[1].axis("off")

                ax[2].imshow(real_images[i])
                ax[2].set_title("Ground Truth")
                ax[2].axis("off")

                plt.show()

                samples_shown += 1

    # Log average metrics
    avg_psnr = np.mean(psnr_values)
    avg_ssim = np.mean(ssim_values)
    print(f"Avg PSNR: {avg_psnr:.4f}, Avg SSIM: {avg_ssim:.4f}")


In [ ]:
# Test the trained model on the validation/test set
test_model_with_metrics(model, val_dl, num_samples=10)
